# Azure Databricks Local Feature Demonstrations
This notebook demonstrates core **Azure Databricks** features locally: **Delta Lake tables**, **Upsert/MERGE**, **Time Travel**, and **Auto Loader** simulation.

## 1. Setup Spark with Delta Lake Support
Make sure Delta Spark SQL extensions and Delta Catalog are loaded.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()
print(f"Spark Active. Catalog configuration: {spark.conf.get('spark.sql.catalog.spark_catalog')}")

## 2. Delta Lake MERGE (Upsert) Operations
Delta tables allow upserting rows using the MERGE statement. This is crucial for updating dimension tables in Medallion architectures.

In [ ]:
import os
delta_path = "s3a://warehouse/wh/delta_customers"

# Create base Delta table
data = [(101, "Alice", "US"), (102, "Bob", "EU")]
df = spark.createDataFrame(data, ["id", "name", "region"])
df.write.format("delta").mode("overwrite").save(delta_path)

# Read back Delta table
spark.read.format("delta").load(delta_path).show()

# Perform MERGE
# In PySpark, we use delta.tables API or Spark SQL directly
spark.sql(f"CREATE TABLE IF NOT EXISTS delta_cust_tb USING DELTA LOCATION '{delta_path}'")

spark.sql("""
MERGE INTO delta_cust_tb AS target
USING (SELECT 102 AS id, 'Robert' AS name, 'EU' AS region UNION SELECT 103 AS id, 'Charlie' AS name, 'AP' AS region) AS source
ON target.id = source.id
WHEN MATCHED THEN UPDATE SET target.name = source.name
WHEN NOT MATCHED THEN INSERT (id, name, region) VALUES (source.id, source.name, source.region)
""")

spark.read.format("delta").load(delta_path).show()

## 3. Delta Lake Time Travel
Delta Lake logs every write operation in the transaction log (`_delta_log`). This allows querying past states of a table.

In [ ]:
# Load historical data by version
print("Querying Delta version 0 (Before Merge):")
try:
    spark.read.format("delta").option("versionAsOf", 0).load(delta_path).show()
    
    print("Querying current Delta version 1 (After Merge):")
    spark.read.format("delta").option("versionAsOf", 1).load(delta_path).show()
except Exception as e:
    print(f"Time travel version unavailable: {e}")

## 4. Databricks Auto Loader Simulation
Auto Loader (`cloudFiles`) processes new files incrementally as they arrive in storage. Locally, we simulate this with Spark Structured Streaming pointing to a file directory path, tracking new file arrivals.

In [ ]:
schema = "id INT, name STRING, score DOUBLE"
source_dir = "s3a://warehouse/autoloader_stage/"
checkpoint_dir = "s3a://warehouse/checkpoints/autoloader/"
target_dir = "s3a://warehouse/wh/autoloader_delta/"

# Simulating the Auto Loader stream reading from CSV directory
autoloader_stream = spark.readStream \
    .format("csv") \
    .option("header", "true") \
    .schema(schema) \
    .load(source_dir)

print("Auto Loader stream pipeline constructed. Ready to scan stage directory.")